# PPQ Data Pipeline

Run this notebook on the DGX (8x A100). It:
1. Sets up paths and repo clones (kokoro, personaplex client).
2. Generates dialogue scripts with an instruct LLM (vLLM server in its own venv,
   with a transformers fallback).
3. Launches one data worker per GPU for TTS + Mimi tokenization (resumable).
4. Prepares Stage-A monologue data from LibriTTS-R (CC-BY-4.0).

Set the paths in the next cell first.

In [ ]:
import os, sys, subprocess, shutil, json, random
from pathlib import Path

# ---- EDIT THESE ----
PPQ_REPO = "/workspace/personaplex-qwen"            # this repo (contains ppq/)
MOSHI_REPO = "/workspace/personaplex"              # original personaplex repo (moshi pkg inside)
MIMI_WEIGHT = "/workspace/models/mimi.safetensors" # moshi mimi codec weights
QWEN_TOKENIZER = "/workspace/models/qwen3-1.7b-base"  # local snapshot of Qwen/Qwen3-1.7B-Base
GEN_MODEL = "/workspace/models/qwen3-8b"          # local snapshot of Qwen/Qwen3-8B (script generator)
DATA_ROOT = Path("/workspace/ppq_data")
N_GPUS = 8
# --------------------

os.environ["PYTHONPATH"] = f"{MOSHI_REPO}/moshi:{PPQ_REPO}:" + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, f"{MOSHI_REPO}/moshi")
sys.path.insert(0, PPQ_REPO)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("data root:", DATA_ROOT)

## 1. Repo clones + dependencies
- kokoro (Apache-2.0) and misaki[en] for TTS.
- vllm in a separate venv to avoid dependency conflicts.


In [ ]:
WORK = Path('/workspace')
KOKORO_DIR = WORK / 'kokoro'
if not KOKORO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/hexgrad/kokoro.git', str(KOKORO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(KOKORO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'misaki[en]', 'soundfile', 'pyloudnorm', 'sphn', 'safetensors', 'datasets', 'huggingface_hub'], check=True)
subprocess.run(['bash', '-c', 'command -v espeak-ng || (apt-get update -qq && apt-get install -y -qq espeak-ng)'], check=False)
import kokoro, misaki
print('kokoro ok:', kokoro.__file__)


In [ ]:
VENV = Path('/workspace/venvs/vllm_gen')
if not (VENV / 'bin' / 'python').exists():
    subprocess.run(['python3', '-m', 'venv', str(VENV)], check=True)
    subprocess.run([str(VENV / 'bin/pip'), 'install', '-q', 'vllm', 'openai'], check=True)
print('vllm venv:', VENV)


## 2. Dialogue script generation
Start a vLLM server with the instruct model and generate ~35k scripts
(35k x ~80 s ~ 800 h; rejected/partial outputs are filtered by schema validation).
The file is resumable: re-run the cell to append more dialogues.


In [ ]:
SCRIPTS = DATA_ROOT / 'scripts.jsonl'
import socket, time
def port_open(port=8000):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', port)) == 0
if not port_open(8000):
    srv_log = open(DATA_ROOT / 'vllm_server.log', 'w')
    subprocess.Popen([
        str(VENV / 'bin/python'), '-m', 'vllm.entrypoints.openai.api_server',
        '--model', GEN_MODEL, '--served-model-name', 'qwen-gen',
        '--tensor-parallel-size', '1', '--max-model-len', '8192',
        '--gpu-memory-utilization', '0.45', '--port', '8000',
    ], stdout=srv_log, stderr=subprocess.STDOUT, env={**os.environ, 'CUDA_VISIBLE_DEVICES': '7'})
    for _ in range(240):
        if port_open(8000):
            break
        time.sleep(5)
assert port_open(8000), 'vLLM server did not start; see vllm_server.log'
print('vLLM server up')


In [ ]:
from ppq.data.script_gen import generate_scripts_openai, generate_scripts_hf
TARGET_SCRIPTS = 35000
existing = sum(1 for _ in open(SCRIPTS)) if SCRIPTS.exists() else 0
print(f'{existing} scripts already present')
if existing < TARGET_SCRIPTS:
    generate_scripts_openai(TARGET_SCRIPTS - existing, SCRIPTS, model='qwen-gen',
                            base_url='http://127.0.0.1:8000/v1', seed=existing)
# Transformers fallback (no vLLM needed):
# generate_scripts_hf(100, SCRIPTS, model_id=GEN_MODEL)
print(f'total scripts: {sum(1 for _ in open(SCRIPTS))}')


In [ ]:
from ppq.data.dialogue import Dialogue
n = n_back = n_int = n_pause = 0
for line in open(SCRIPTS):
    d = Dialogue.from_dict(json.loads(line))
    n += 1
    n_back += sum(t.backchannel for t in d.turns)
    n_int += sum(t.interrupt for t in d.turns)
    n_pause += sum(t.pause_mid for t in d.turns)
print(f'{n} dialogues | backchannels {n_back} | interrupts {n_int} | mid-pauses {n_pause}')


## 3. TTS + stereo stitching + Mimi tokenization (one worker per GPU)
Each worker gets a disjoint subset of the scripts and its own GPU.
All workers are resumable: re-running the cell skips finished dialogues.
First build a voice prompt for the agent (first agent_voice_duration seconds
of a Kokoro agent voice, saved as a 24 kHz wav).


In [ ]:
import soundfile as sf
from ppq.data.tts_kokoro import KokoroTTS
from ppq.config import PPQModelConfig
cfg = PPQModelConfig()
VOICE_PROMPT = DATA_ROOT / 'voice_prompt_agent.wav'
if not VOICE_PROMPT.exists():
    tts = KokoroTTS()
    res = tts.synth(
        'Hello there! It is lovely to chat with you today. '
        'I am always happy to help with whatever is on your mind, '
        'so feel free to ask me anything at all.', voice='af_heart')
    n = int(cfg.voice_prompt_duration * 24000)
    sf.write(VOICE_PROMPT, res.wav[:n], 24000)
print('voice prompt:', VOICE_PROMPT, f'({cfg.voice_prompt_duration}s)')


In [ ]:
TOKENS_DIR = DATA_ROOT / 'tokens'
TOKENS_DIR.mkdir(exist_ok=True)
procs = []
for gpu in range(N_GPUS):
    log = open(DATA_ROOT / f'worker_synth_{gpu}.log', 'w')
    p = subprocess.Popen([
        sys.executable, '-m', 'ppq.data.worker',
        '--stage', 'synth',
        '--shard', str(gpu), '--num-shards', str(N_GPUS),
        '--scripts', str(SCRIPTS),
        '--out', str(TOKENS_DIR),
        '--mimi-weight', MIMI_WEIGHT,
        '--tokenizer', QWEN_TOKENIZER,
        '--voice-prompt', str(VOICE_PROMPT),
        '--seed', '0',
    ], stdout=log, stderr=subprocess.STDOUT,
       env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)})
    procs.append(p)
print('launched', len(procs), 'synth workers')


In [ ]:
for p in procs:
    p.wait()
print('synth workers done:', [p.returncode for p in procs])


In [ ]:
import glob
tot = {'n': 0, 'seconds': 0.0, 'backchannels': 0, 'interrupts': 0, 'mid_pauses': 0, 'words_estimated': 0}
for mf in glob.glob(str(TOKENS_DIR / 'manifest-*.jsonl')):
    for line in open(mf):
        e = json.loads(line)
        tot['n'] += 1
        tot['seconds'] += e['duration_s']
        tot['backchannels'] += e['backchannels']
        tot['interrupts'] += e['interrupts']
        tot['mid_pauses'] += e['mid_pauses']
        tot['words_estimated'] += int(e['words_estimated'])
print(json.dumps({'dialogues': tot['n'], 'hours': round(tot['seconds'] / 3600, 1),
    'backchannels': tot['backchannels'], 'interrupts': tot['interrupts'],
    'mid_pauses': tot['mid_pauses'],
    'frac_words_estimated': round(tot['words_estimated'] / max(1, tot['n']), 3)}, indent=1))


## 4. Stage A: LibriTTS-R monologues (~100 h, CC-BY-4.0)


In [ ]:
procs_lt = []
for gpu in range(N_GPUS):
    log = open(DATA_ROOT / f'worker_libritts_{gpu}.log', 'w')
    p = subprocess.Popen([
        sys.executable, '-m', 'ppq.data.worker',
        '--stage', 'libritts',
        '--shard', str(gpu), '--num-shards', str(N_GPUS),
        '--out', str(TOKENS_DIR),
        '--mimi-weight', MIMI_WEIGHT,
        '--max-hours', '100',
    ], stdout=log, stderr=subprocess.STDOUT,
       env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)})
    procs_lt.append(p)
for p in procs_lt:
    p.wait()
print('libritts workers done:', [p.returncode for p in procs_lt])


## 5. PersonaPlex web client (for serving later)
The PPQ server speaks the stock PersonaPlex protocol, so we reuse its web UI.
Build it here so the training notebook can serve it.


In [ ]:
CLIENT_DIR = Path(MOSHI_REPO) / 'client'
if not (CLIENT_DIR / 'dist').exists():
    subprocess.run(['npm', 'install'], cwd=CLIENT_DIR, check=True)
    subprocess.run(['npm', 'run', 'build'], cwd=CLIENT_DIR, check=True)
print('client dist:', CLIENT_DIR / 'dist')
